# BKMeeting AI Hub Option 1 NPU Pilots

This notebook is the day-to-day operator notebook for the `Option 1` pilot flow.

Scope of this notebook:

- stay fully in `python-model-test`
- do not touch Android packaging
- do not run Phase 4 gate logic
- do not run Phase 5 packaging logic
- support the common research loop:
  - prepare
  - compile or reuse target
  - run on cloud device
  - optional debug inspection
  - hybrid e2e compare


## Environment Notes

Before running this notebook, make sure the current environment can already execute the local `python-model-test` bundle helpers.

Minimum practical dependencies:

- `qai-hub`
- `torch`
- `torchaudio`
- `numpy`
- local editable install of this repo if needed

The Zipformer pilot uses the existing repo feature-extraction path, so `torchaudio` must be available.

Environment setup is intentionally outside the normal execution flow of this notebook.
If you still need one-time dependency bootstrap, do that before opening the notebook.


In [ ]:
from pathlib import Path
import os
import sys

import qai_hub as hub

sys.path.insert(0, str(Path.cwd() / "src"))

from tools.aihub_option1_pilots import resolve_qai_hub_api_token

API_TOKEN = resolve_qai_hub_api_token(repo_root=Path.cwd())

if not API_TOKEN:
    print("Set QAI_HUB_API_TOKEN in .env or your shell environment before running Qualcomm AI Hub configuration.")
else:
    os.environ["QAI_HUB_API_TOKEN"] = API_TOKEN
    available_devices = hub.get_devices()
    print("Loaded QAI_HUB_API_TOKEN from .env or shell environment.")
    print("AI Hub device count:", len(available_devices))
    print("AI Hub first devices:")
    for device in available_devices[:5]:
        print(device)


In [ ]:
import sys
from pathlib import Path

import onnxruntime as ort
import qai_hub as hub

from model_bundle.fixtures import read_jsonl
sys.path.insert(0, str(Path.cwd() / "src"))

from tools.aihub_option1_hybrid_pipeline import (
    run_vpcd_hybrid_evaluation,
    run_zipformer_hybrid_evaluation,
)
from tools.aihub_option1_pilots import (
    build_compile_options,
    build_job_options,
    build_option1_runtime_config,
    build_vpcd_autoregressive_calibration_entries,
    build_vpcd_input_specs,
    build_vpcd_single_step_inputs,
    build_zipformer_encoder_inference_entries,
    build_zipformer_encoder_input_specs,
    compare_output_tensors,
    coerce_inputs_for_compiled_model,
    prepare_vpcd_option1_source_model,
    prepare_zipformer_encoder_option1_source_model,
    resolve_target_model_id,
    resolve_vpcd_aihub_quantize_dtype_names,
    resolve_vpcd_fp32_source_model_path,
    resolve_vpcd_pilot_source,
    resolve_zipformer_encoder_pilot_source,
    summarize_vpcd_step_logits,
    write_compile_run_record,
    write_live_run_record,
    write_prepared_artifact_record,
    wrap_single_inference_inputs,
)


In [ ]:
DEVICE_NAME = "Samsung Galaxy S24 (Family)"
QAIRT_VERSION = None
RUN_LABEL = "20260513-1am"

ENABLE_ZIPFORMER = True
ENABLE_VPCD = True
ENABLE_PROFILE_DURING_RUN = False
ENABLE_DEBUG_OUTPUT_INSPECTION = False

# Set either pilot flag to False when you want to skip that pilot entirely.
# All later code cells for that pilot become no-ops and the shared summary cell ignores it.

# Use one stable RUN_LABEL per compiled artifact set.
# Keep the same RUN_LABEL when you want to reuse an earlier compile without recompiling.

ZIPFORMER_TARGET_MODEL_ID = None
VPCD_TARGET_MODEL_ID = None
AUTO_SKIP_COMPILE_IF_RECORD_EXISTS = True

ZIPFORMER_HYBRID_MAX_SAMPLES = 2
VPCD_HYBRID_MAX_SAMPLES = 4
VPCD_CALIBRATION_MAX_SAMPLES = 24
VPCD_CALIBRATION_MAX_GENERATION_LENGTH = 32
VPCD_CALIBRATION_SOURCE_PATH = Path("build/calibration/vlsp2020/vpcd_transcriptions.txt")

RUNTIME_CONFIG = build_option1_runtime_config(
    device_name=DEVICE_NAME,
    qairt_version=QAIRT_VERSION,
    repo_root=Path.cwd(),
)
job_options = build_job_options(
    compute_unit=RUNTIME_CONFIG.compute_unit,
    qairt_version=RUNTIME_CONFIG.qairt_version,
)

print("device:", RUNTIME_CONFIG.device_name)
print("qairt_version:", RUNTIME_CONFIG.qairt_version)
print("artifact_root:", RUNTIME_CONFIG.artifact_root)
print("record_root:", RUNTIME_CONFIG.record_root)
print("job_options:", job_options)
print("run_label:", RUN_LABEL)
print("enable zipformer:", ENABLE_ZIPFORMER)
print("enable vpcd:", ENABLE_VPCD)
print("enable profile during run:", ENABLE_PROFILE_DURING_RUN)
print("enable debug output inspection:", ENABLE_DEBUG_OUTPUT_INSPECTION)
print("zipformer reuse target model id:", ZIPFORMER_TARGET_MODEL_ID)
print("vpcd reuse target model id:", VPCD_TARGET_MODEL_ID)
print("auto skip compile if record exists:", AUTO_SKIP_COMPILE_IF_RECORD_EXISTS)
print("zipformer hybrid max samples:", ZIPFORMER_HYBRID_MAX_SAMPLES)
print("vpcd hybrid max samples:", VPCD_HYBRID_MAX_SAMPLES)
print("vpcd calibration max samples:", VPCD_CALIBRATION_MAX_SAMPLES)
print("vpcd calibration max generation length:", VPCD_CALIBRATION_MAX_GENERATION_LENGTH)
print("vpcd calibration source override:", VPCD_CALIBRATION_SOURCE_PATH)


## How To Use This Notebook

This notebook supports two normal workflows.

### Workflow A: Compile From Scratch

Use this when you do **not** already have a compiled target model for the current pilot.

1. Run setup and config.
2. Keep `*_TARGET_MODEL_ID = None`.
3. Choose a stable `RUN_LABEL`.
4. For each enabled pilot, run:
   - `Prepare`
   - `Compile Only`
   - `Resolve Existing Compiled Target`
   - `Run And Compare Against The Compiled Target`
   - optional `Output Inspection (Debug Only)`
   - `Hybrid E2E Run`
   - `Final Compare`

### Workflow B: Reuse An Existing Compiled Target

Use this when compile already succeeded earlier and you only want to rerun inference and correctness checks.

1. Keep the same `RUN_LABEL` and leave `*_TARGET_MODEL_ID = None`, or paste a known target model id.
2. Skip `Compile Only`.
3. For each enabled pilot, run:
   - `Prepare`
   - `Resolve Existing Compiled Target`
   - `Run And Compare Against The Compiled Target`
   - optional `Output Inspection (Debug Only)`
   - `Hybrid E2E Run`
   - `Final Compare`

Notes:

- `ENABLE_PROFILE_DURING_RUN = False` keeps normal output checks fast.
- `ENABLE_DEBUG_OUTPUT_INSPECTION = True` enables the tensor-level diagnostic sections.


## Pilot 1: Zipformer Encoder-First

This pilot targets the first ASR slice that BKMeeting wants to offload first: the encoder graph.

The current local helper now prepares a dedicated AI Hub upload artifact from the fixed-shape encoder source.

- base source: fixed-shape encoder ONNX
- upload artifact: ORT-optimized + symbolic-shape-prepared + HTP bool-slice rewrite
- local fixtures: current Zipformer bundle sample manifest
- current verified lane: direct `submit_compile_job(...)` on the prepared source model


In [ ]:
if ENABLE_ZIPFORMER:
    zipformer_pilot_name = "zipformer_encoder_option1"
    zipformer_source = resolve_zipformer_encoder_pilot_source(RUNTIME_CONFIG.repo_root)
    zipformer_source_model_path = prepare_zipformer_encoder_option1_source_model(
        zipformer_source,
        output_path=RUNTIME_CONFIG.pilot_artifact_dir(zipformer_pilot_name) / "encoder.aihub.option1.onnx",
    )
    zipformer_input_specs = build_zipformer_encoder_input_specs(zipformer_source)
    zipformer_compile_options = build_compile_options(
        qairt_version=RUNTIME_CONFIG.qairt_version,
        input_specs=zipformer_input_specs,
    )
    zipformer_raw_inference_inputs = build_zipformer_encoder_inference_entries(zipformer_source)
    zipformer_inference_inputs = coerce_inputs_for_compiled_model(
        zipformer_raw_inference_inputs,
        input_specs=zipformer_input_specs,
    )
    zipformer_prepared_record_path = write_prepared_artifact_record(
        pilot_name=zipformer_pilot_name,
        runtime_config=RUNTIME_CONFIG,
        source_model_path=zipformer_source.source_model_path,
        prepared_model_path=zipformer_source_model_path,
        input_specs=zipformer_input_specs,
        compile_options=zipformer_compile_options,
        run_label=RUN_LABEL,
    )

    print("zipformer base source model:", zipformer_source.source_model_path)
    print("zipformer prepared upload model:", zipformer_source_model_path)
    print("zipformer bundle manifest:", zipformer_source.bundle_manifest_path)
    print("zipformer input specs:", zipformer_input_specs)
    print("zipformer compile options:", zipformer_compile_options)
    print("zipformer prepared record:", zipformer_prepared_record_path)
    print({name: [value.shape for value in values] for name, values in zipformer_inference_inputs.items()})
else:
    print('Skipping Zipformer cell 8 because ENABLE_ZIPFORMER is False.')


### Zipformer Compile Only

Use this section only when you need to create a **new compiled target model** for Zipformer.

Run this section when:

- this is your first time testing Zipformer on the selected cloud device
- you changed the prepared source model or compile options
- you want a fresh compiled artifact under a new `RUN_LABEL`

After this cell succeeds, save or remember at least one of these:

- `RUN_LABEL`
- `zipformer target model id`
- the record file `build/aihub/records/zipformer_encoder_option1/compile-run-<RUN_LABEL>.json`

If you only want to rerun inference and compare outputs, do **not** rerun this section. Jump to `Resolve Existing Compiled Target` instead.


In [ ]:
if ENABLE_ZIPFORMER:
    zipformer_compile_record_target = RUNTIME_CONFIG.pilot_record_dir(zipformer_pilot_name) / f"compile-run-{RUN_LABEL}.json"
    zipformer_should_compile = not (
        AUTO_SKIP_COMPILE_IF_RECORD_EXISTS
        and ZIPFORMER_TARGET_MODEL_ID is None
        and zipformer_compile_record_target.exists()
    )
    zipformer_compile_job = None
    zipformer_compiled_target_model = None
    zipformer_compile_record_path = zipformer_compile_record_target

    if ZIPFORMER_TARGET_MODEL_ID is not None:
        print("Skipping Zipformer compile because ZIPFORMER_TARGET_MODEL_ID is set.")
    elif zipformer_should_compile:
        zipformer_compile_job = hub.submit_compile_job(
            model=zipformer_source_model_path,
            device=hub.Device(RUNTIME_CONFIG.device_name),
            input_specs=zipformer_input_specs,
            options=zipformer_compile_options,
            name="bkmeeting-zipformer-encoder-precompiled-qnn-onnx",
        )
        zipformer_compiled_target_model = zipformer_compile_job.get_target_model()
        zipformer_compile_record_path = write_compile_run_record(
            pilot_name=zipformer_pilot_name,
            runtime_config=RUNTIME_CONFIG,
            compile_options=zipformer_compile_options,
            compile_job=zipformer_compile_job,
            target_model=zipformer_compiled_target_model,
            run_label=RUN_LABEL,
        )

        print("zipformer compile job:", zipformer_compile_job.url)
        print("zipformer target model id:", zipformer_compiled_target_model.model_id)
        print("zipformer target model url:", zipformer_compiled_target_model.url)
        print("zipformer compile record:", zipformer_compile_record_path)
    else:
        print("Skipping Zipformer compile because compile record already exists:", zipformer_compile_record_target)
else:
    print('Skipping Zipformer cell 10 because ENABLE_ZIPFORMER is False.')


### Resolve Existing Compiled Target

This section decides **which compiled Zipformer target model** will be used for profile, inference, and comparison.

It works in two modes:

1. `ZIPFORMER_TARGET_MODEL_ID = None`
   - the notebook reads `build/aihub/records/zipformer_encoder_option1/compile-run-<RUN_LABEL>.json`
   - use this when you want to reuse a previous compile by label
2. `ZIPFORMER_TARGET_MODEL_ID = "..."`
   - the notebook skips record lookup and uses that exact model id directly
   - use this when you copied a target model id from an earlier notebook run or AI Hub page

If this cell fails with a missing record error, it usually means one of these:

- you never ran `Compile Only` for this `RUN_LABEL`
- you changed `RUN_LABEL` and the matching `compile-run-<RUN_LABEL>.json` does not exist yet
- you should paste a known `ZIPFORMER_TARGET_MODEL_ID` manually


In [ ]:
if ENABLE_ZIPFORMER:
    zipformer_target_model_id = resolve_target_model_id(
        pilot_name=zipformer_pilot_name,
        runtime_config=RUNTIME_CONFIG,
        explicit_target_model_id=ZIPFORMER_TARGET_MODEL_ID,
        run_label=RUN_LABEL,
    )
    zipformer_target_model = hub.get_model(zipformer_target_model_id)

    print("zipformer resolved target model id:", zipformer_target_model_id)
    print("zipformer target model url:", zipformer_target_model.url)
else:
    print('Skipping Zipformer cell 12 because ENABLE_ZIPFORMER is False.')


### Run And Compare Against The Compiled Target

This is the **fast rerun loop** for Zipformer.

Use this section when:

- compile already exists and you want to rerun on the cloud NPU device
- you want fresh profile/inference jobs without paying compile time again
- you want to compare cloud output against the local CPU baseline again

This section does three things:

1. profile the already-compiled target model on the selected cloud device
2. run inference on the same compiled target model
3. write a fresh `live-run-<RUN_LABEL>.json` record and leave `zipformer_output` ready for the inspection cell

After this cell finishes, run the `Zipformer Output Inspection` cell right below it.


In [ ]:
if ENABLE_ZIPFORMER:
    zipformer_profile_job = None
    zipformer_profile = None
    if ENABLE_PROFILE_DURING_RUN:
        zipformer_profile_job = hub.submit_profile_job(
            model=zipformer_target_model,
            device=hub.Device(RUNTIME_CONFIG.device_name),
            options=job_options,
            name="bkmeeting-zipformer-encoder-profile-npu",
        )
        zipformer_profile = zipformer_profile_job.download_profile()

    zipformer_inference_job = hub.submit_inference_job(
        model=zipformer_target_model,
        device=hub.Device(RUNTIME_CONFIG.device_name),
        inputs=zipformer_inference_inputs,
        options=job_options,
        name="bkmeeting-zipformer-encoder-inference-npu",
    )
    zipformer_output = zipformer_inference_job.download_output_data()
    zipformer_live_record_path = write_live_run_record(
        pilot_name=zipformer_pilot_name,
        runtime_config=RUNTIME_CONFIG,
        compile_options=zipformer_compile_options,
        job_options=job_options,
        compile_job=zipformer_compile_job if "zipformer_compile_job" in globals() else {"status": "reused-target-model"},
        profile_job=zipformer_profile_job,
        inference_job=zipformer_inference_job,
        output_tensors=zipformer_output,
        run_label=RUN_LABEL,
    )

    print("zipformer profile job:", zipformer_profile_job.url if zipformer_profile_job is not None else "skipped")
    print("zipformer inference job:", zipformer_inference_job.url)
    print("zipformer live record:", zipformer_live_record_path)
    print("zipformer output tensors:", {name: [value.shape for value in values] for name, values in zipformer_output.items()})
else:
    print('Skipping Zipformer cell 14 because ENABLE_ZIPFORMER is False.')


## Zipformer Output Inspection (Debug Only)

This section checks encoder tensors only.
Use it when `ENABLE_DEBUG_OUTPUT_INSPECTION = True` and you want a tensor-level sanity check before the slower hybrid transcript path.

Do **not** treat this as the final correctness gate.
The final transcript comparison happens in `Zipformer Hybrid E2E Run` and `Zipformer Final Compare Against Expected Outputs`.


In [ ]:
if ENABLE_ZIPFORMER and ENABLE_DEBUG_OUTPUT_INSPECTION:
    zipformer_cpu_inputs = {name: values[0] for name, values in zipformer_raw_inference_inputs.items()}
    zipformer_cpu_session = ort.InferenceSession(
        zipformer_source.source_model_path.as_posix(),
        providers=["CPUExecutionProvider"],
    )
    zipformer_cpu_output_arrays = zipformer_cpu_session.run(None, zipformer_cpu_inputs)
    zipformer_cpu_output = {f"output_{index}": [value] for index, value in enumerate(zipformer_cpu_output_arrays)}
    zipformer_output_comparison = compare_output_tensors(
        zipformer_cpu_output,
        zipformer_output,
        atol=1e-3,
        rtol=1e-3,
    )
    zipformer_expected_outputs = read_jsonl(zipformer_source.bundle_manifest_path.parent / "expected_outputs.jsonl")

    print("zipformer reference transcript:", zipformer_expected_outputs[0]["text"] if zipformer_expected_outputs else "n/a")
    print("zipformer encoder_out_lens (cloud):", zipformer_output["output_1"][0].tolist())
    print("zipformer encoder frame preview (cloud):")
    print(zipformer_output["output_0"][0][0, :2, :8])
    zipformer_output_comparison
elif ENABLE_ZIPFORMER:
    print('Skipping Zipformer output inspection because ENABLE_DEBUG_OUTPUT_INSPECTION is False.')
else:
    print('Skipping Zipformer cell 16 because ENABLE_ZIPFORMER is False.')


### Zipformer Hybrid E2E Run

Run this section only after the compiled target has already been resolved.
This is the first point where the notebook executes the real Phase 3 hybrid pipeline:

1. feature extraction on the host
2. encoder inference on the compiled cloud NPU target
3. greedy decoder and joiner on the host CPU
4. write `hybrid-run-<RUN_LABEL>.json` under `build/aihub/records/zipformer_hybrid_option1/`


In [ ]:
if ENABLE_ZIPFORMER:
    zipformer_hybrid_report = run_zipformer_hybrid_evaluation(
        runtime_config=RUNTIME_CONFIG,
        run_label=RUN_LABEL,
        explicit_target_model_id=ZIPFORMER_TARGET_MODEL_ID,
        max_samples=ZIPFORMER_HYBRID_MAX_SAMPLES,
    )
    zipformer_hybrid_record_path = zipformer_hybrid_report["record_path"]

    print("zipformer hybrid target model id:", zipformer_hybrid_report["target_reference"].target_model_id)
    print("zipformer hybrid summary:", zipformer_hybrid_report["summary"])
    print("zipformer hybrid record:", zipformer_hybrid_record_path)
else:
    print('Skipping Zipformer cell 18 because ENABLE_ZIPFORMER is False.')


### Zipformer Final Compare Against Expected Outputs

This is the final correctness gate for Zipformer in this notebook.
Only this section decides whether the evaluated samples match `expected_outputs.jsonl` end to end.


In [ ]:
if ENABLE_ZIPFORMER:
    zipformer_hybrid_results = zipformer_hybrid_report["results"]
    zipformer_hybrid_comparable = [row for row in zipformer_hybrid_results if row["matches_expected"] is not None]
    zipformer_hybrid_mismatches = [row for row in zipformer_hybrid_comparable if not row["matches_expected"]]
    zipformer_hybrid_unavailable = [row for row in zipformer_hybrid_results if row["matches_expected"] is None]

    print("zipformer final transcript compare:")
    for row in zipformer_hybrid_results:
        print(
            {
                "sample_id": row["sample_id"],
                "audio_path": row["audio_path"],
                "text": row["text"],
                "expected_text": row["expected_text"],
                "expected_available": row["expected_available"],
                "matches_expected": row["matches_expected"],
                "cloud_inference_seconds": row["cloud_inference_seconds"],
                "decode_seconds": row["decode_seconds"],
            }
        )

    if zipformer_hybrid_unavailable:
        print("zipformer rows without expected transcript fixture:")
        for row in zipformer_hybrid_unavailable:
            print({"sample_id": row["sample_id"], "audio_path": row["audio_path"]})

    if zipformer_hybrid_mismatches:
        print("zipformer mismatches:")
        for row in zipformer_hybrid_mismatches:
            print(
                {
                    "sample_id": row["sample_id"],
                    "audio_path": row["audio_path"],
                    "text": row["text"],
                    "expected_text": row["expected_text"],
                }
            )
    elif zipformer_hybrid_comparable:
        print("zipformer all comparable samples matched expected transcripts.")
    else:
        print("zipformer final compare could not run because no expected transcript fixtures were available.")
else:
    print('Skipping Zipformer cell 20 because ENABLE_ZIPFORMER is False.')


## Pilot 2: VPCD Model-Session-First

This pilot targets the punctuation model session while keeping tokenization on the host side.

Important current caveats:

- prefer the repo FP32 export when available, then freeze it to the fixed bundle shapes before upload
- if the source is still QDQ after preparation, compile it directly as the pragmatic fallback lane
- compiled inference inputs must be coerced from `int64` to `int32` when `--truncate_64bit_io` is present

The compile path now prefers autoregressive calibration derived from the FP32 baseline over real text samples, because the earlier single-step-only calibration produced unstable cloud logits.

The canonical VPCD quantize recipe now comes from `src/quantize/projects/vpcd.py`. This notebook only resolves sources, uploads to AI Hub, and runs cloud jobs; it no longer owns the activation/weight dtype policy.


In [ ]:
if ENABLE_VPCD:
    vpcd_pilot_name = "vpcd_option1"
    vpcd_source = resolve_vpcd_pilot_source(RUNTIME_CONFIG.repo_root)
    vpcd_original_source_model_path = resolve_vpcd_fp32_source_model_path(vpcd_source) or vpcd_source.model_path
    vpcd_prepared_source_model_path, vpcd_is_quantized_source = prepare_vpcd_option1_source_model(
        vpcd_source,
        output_path=RUNTIME_CONFIG.pilot_artifact_dir(vpcd_pilot_name) / "model.option1.onnx",
    )
    vpcd_input_specs = build_vpcd_input_specs(vpcd_source)
    vpcd_quantize_dtype_names = resolve_vpcd_aihub_quantize_dtype_names(vpcd_source)
    vpcd_compile_options = build_compile_options(
        qairt_version=RUNTIME_CONFIG.qairt_version,
        input_specs=vpcd_input_specs,
    )
    vpcd_single_step_inputs = build_vpcd_single_step_inputs(vpcd_source, sample_index=0)
    vpcd_raw_inference_inputs = wrap_single_inference_inputs(vpcd_single_step_inputs)
    vpcd_inference_inputs = coerce_inputs_for_compiled_model(
        vpcd_raw_inference_inputs,
        input_specs=vpcd_input_specs,
    )
    vpcd_prepared_record_path = write_prepared_artifact_record(
        pilot_name=vpcd_pilot_name,
        runtime_config=RUNTIME_CONFIG,
        source_model_path=vpcd_original_source_model_path,
        prepared_model_path=vpcd_prepared_source_model_path,
        input_specs=vpcd_input_specs,
        compile_options=vpcd_compile_options,
        run_label=RUN_LABEL,
    )

    print("vpcd source model:", vpcd_original_source_model_path)
    print("vpcd prepared upload model:", vpcd_prepared_source_model_path)
    print("vpcd input specs:", vpcd_input_specs)
    print("vpcd compile options:", vpcd_compile_options)
    print("vpcd preferred quantize dtypes:", vpcd_quantize_dtype_names)
    print("vpcd quantize source of truth:", "src/quantize/projects/vpcd.py")
    print("vpcd quantized source:", vpcd_is_quantized_source)
    print("vpcd calibration: lazy build in Compile Only")
    print("vpcd prepared record:", vpcd_prepared_record_path)
    print({name: [value.shape for value in values] for name, values in vpcd_inference_inputs.items()})
else:
    print('Skipping VPCD cell 22 because ENABLE_VPCD is False.')


### VPCD Compile Only

Use this section only when you need to create a **new compiled target model** for VPCD.

Run this section when:

- this is your first time testing VPCD on the selected cloud device
- you changed the prepared source model, quantize step, or compile options
- you want a fresh compiled artifact under a new `RUN_LABEL`

After this cell succeeds, save or remember at least one of these:

- `RUN_LABEL`
- `vpcd target model id`
- the record file `build/aihub/records/vpcd_option1/compile-run-<RUN_LABEL>.json`

If you only want to rerun inference and compare outputs, do **not** rerun this section. Jump to `Resolve Existing Compiled Target` instead.


In [ ]:
if ENABLE_VPCD:
    vpcd_compile_record_target = RUNTIME_CONFIG.pilot_record_dir(vpcd_pilot_name) / f"compile-run-{RUN_LABEL}.json"
    vpcd_should_compile = not (
        AUTO_SKIP_COMPILE_IF_RECORD_EXISTS
        and VPCD_TARGET_MODEL_ID is None
        and vpcd_compile_record_target.exists()
    )
    vpcd_quantize_job = None
    vpcd_compile_job = None
    vpcd_compiled_target_model = None
    vpcd_compile_record_path = vpcd_compile_record_target

    if VPCD_TARGET_MODEL_ID is not None:
        print("Skipping VPCD compile because VPCD_TARGET_MODEL_ID is set.")
    elif vpcd_should_compile:
        if vpcd_is_quantized_source:
            vpcd_compile_input_model = vpcd_prepared_source_model_path
            print("VPCD source is already QDQ. Compiling directly for the current AI Hub pilot.")
        else:
            vpcd_calibration_data, vpcd_calibration_stats = build_vpcd_autoregressive_calibration_entries(
                vpcd_source,
                calibration_source_path=VPCD_CALIBRATION_SOURCE_PATH,
                max_samples=VPCD_CALIBRATION_MAX_SAMPLES,
                max_generation_length=VPCD_CALIBRATION_MAX_GENERATION_LENGTH,
                ort_provider="cpu",
            )
            print("vpcd calibration stats:", vpcd_calibration_stats)
            vpcd_quantize_job = hub.submit_quantize_job(
                model=vpcd_prepared_source_model_path,
                calibration_data=vpcd_calibration_data,
                weights_dtype=getattr(hub.QuantizeDtype, vpcd_quantize_dtype_names["weights_dtype_name"]),
                activations_dtype=getattr(hub.QuantizeDtype, vpcd_quantize_dtype_names["activations_dtype_name"]),
                name="bkmeeting-vpcd-quantize",
            )
            vpcd_compile_input_model = vpcd_quantize_job.get_target_model()
            print("vpcd quantize job:", vpcd_quantize_job.url)
            print("vpcd quantize dtypes:", vpcd_quantize_dtype_names)

        vpcd_compile_job = hub.submit_compile_job(
            model=vpcd_compile_input_model,
            device=hub.Device(RUNTIME_CONFIG.device_name),
            input_specs=vpcd_input_specs,
            options=vpcd_compile_options,
            name="bkmeeting-vpcd-precompiled-qnn-onnx",
        )
        vpcd_compiled_target_model = vpcd_compile_job.get_target_model()
        vpcd_compile_record_path = write_compile_run_record(
            pilot_name=vpcd_pilot_name,
            runtime_config=RUNTIME_CONFIG,
            compile_options=vpcd_compile_options,
            compile_job=vpcd_compile_job,
            target_model=vpcd_compiled_target_model,
            run_label=RUN_LABEL,
        )

        print("vpcd compile job:", vpcd_compile_job.url)
        print("vpcd target model id:", vpcd_compiled_target_model.model_id)
        print("vpcd target model url:", vpcd_compiled_target_model.url)
        print("vpcd compile record:", vpcd_compile_record_path)
    else:
        print("Skipping VPCD compile because compile record already exists:", vpcd_compile_record_target)
else:
    print('Skipping VPCD cell 24 because ENABLE_VPCD is False.')


### Resolve Existing Compiled Target

This section decides **which compiled VPCD target model** will be used for profile, inference, and comparison.

It works in two modes:

1. `VPCD_TARGET_MODEL_ID = None`
   - the notebook reads `build/aihub/records/vpcd_option1/compile-run-<RUN_LABEL>.json`
   - use this when you want to reuse a previous compile by label
2. `VPCD_TARGET_MODEL_ID = "..."`
   - the notebook skips record lookup and uses that exact model id directly
   - use this when you copied a target model id from an earlier notebook run or AI Hub page

If this cell fails with a missing record error, it usually means one of these:

- you never ran `Compile Only` for this `RUN_LABEL`
- you changed `RUN_LABEL` and the matching `compile-run-<RUN_LABEL>.json` does not exist yet
- you should paste a known `VPCD_TARGET_MODEL_ID` manually


In [ ]:
if ENABLE_VPCD:
    vpcd_target_model_id = resolve_target_model_id(
        pilot_name=vpcd_pilot_name,
        runtime_config=RUNTIME_CONFIG,
        explicit_target_model_id=VPCD_TARGET_MODEL_ID,
        run_label=RUN_LABEL,
    )
    vpcd_target_model = hub.get_model(vpcd_target_model_id)

    print("vpcd resolved target model id:", vpcd_target_model_id)
    print("vpcd target model url:", vpcd_target_model.url)
else:
    print('Skipping VPCD cell 26 because ENABLE_VPCD is False.')


### Run And Compare Against The Compiled Target

This is the **fast rerun loop** for VPCD.

Use this section when:

- compile already exists and you want to rerun on the cloud NPU device
- you want fresh profile/inference jobs without paying compile time again
- you want to compare cloud output against the local CPU baseline again

This section does three things:

1. profile the already-compiled target model on the selected cloud device
2. run inference on the same compiled target model
3. write a fresh `live-run-<RUN_LABEL>.json` record and leave `vpcd_output` ready for the inspection cell

After this cell finishes, run the `VPCD Output Inspection` cell right below it.


In [ ]:
if ENABLE_VPCD:
    vpcd_profile_job = None
    vpcd_profile = None
    if ENABLE_PROFILE_DURING_RUN:
        vpcd_profile_job = hub.submit_profile_job(
            model=vpcd_target_model,
            device=hub.Device(RUNTIME_CONFIG.device_name),
            options=job_options,
            name="bkmeeting-vpcd-profile-npu",
        )
        vpcd_profile = vpcd_profile_job.download_profile()

    vpcd_inference_job = hub.submit_inference_job(
        model=vpcd_target_model,
        device=hub.Device(RUNTIME_CONFIG.device_name),
        inputs=vpcd_inference_inputs,
        options=job_options,
        name="bkmeeting-vpcd-inference-npu",
    )
    vpcd_output = vpcd_inference_job.download_output_data()
    vpcd_live_record_path = write_live_run_record(
        pilot_name=vpcd_pilot_name,
        runtime_config=RUNTIME_CONFIG,
        compile_options=vpcd_compile_options,
        job_options=job_options,
        compile_job=vpcd_compile_job if "vpcd_compile_job" in globals() else {"status": "reused-target-model"},
        profile_job=vpcd_profile_job,
        inference_job=vpcd_inference_job,
        output_tensors=vpcd_output,
        run_label=RUN_LABEL,
    )

    print("vpcd profile job:", vpcd_profile_job.url if vpcd_profile_job is not None else "skipped")
    print("vpcd inference job:", vpcd_inference_job.url)
    print("vpcd live record:", vpcd_live_record_path)
    print("vpcd output tensors:", {name: [value.shape for value in values] for name, values in vpcd_output.items()})
else:
    print('Skipping VPCD cell 28 because ENABLE_VPCD is False.')


## VPCD Output Inspection (Debug Only)

This section checks one model-step tensor only.
Use it when `ENABLE_DEBUG_OUTPUT_INSPECTION = True` and you want a logits-level sanity check before the full hybrid decode loop.

Do **not** treat this as the final correctness gate.
The final punctuation comparison happens in `VPCD Hybrid E2E Run` and `VPCD Final Compare Against Gold Samples`.


In [ ]:
if ENABLE_VPCD and ENABLE_DEBUG_OUTPUT_INSPECTION:
    vpcd_cpu_inputs = {name: value for name, value in vpcd_single_step_inputs.items()}
    vpcd_cpu_session = ort.InferenceSession(
        vpcd_prepared_source_model_path.as_posix(),
        providers=["CPUExecutionProvider"],
    )
    vpcd_cpu_output_arrays = vpcd_cpu_session.run(None, vpcd_cpu_inputs)
    vpcd_cpu_output = {f"output_{index}": [value] for index, value in enumerate(vpcd_cpu_output_arrays)}
    vpcd_output_comparison = compare_output_tensors(
        vpcd_cpu_output,
        vpcd_output,
        atol=1e-2,
        rtol=1e-2,
    )
    vpcd_golden_sample = read_jsonl(vpcd_source.golden_samples_path)[0]
    vpcd_cpu_next_token_summary = summarize_vpcd_step_logits(
        vpcd_cpu_output["output_0"][0],
        vpcd_single_step_inputs["decoder_attention_mask"],
        top_k=5,
    )
    vpcd_next_token_summary = summarize_vpcd_step_logits(
        vpcd_output["output_0"][0],
        vpcd_single_step_inputs["decoder_attention_mask"],
        top_k=5,
    )

    print("vpcd raw_text:", vpcd_golden_sample["raw_text"])
    print("vpcd expected_output:", vpcd_golden_sample["expected_output"])
    print("vpcd active decoder index:", vpcd_next_token_summary["active_index"])
    print("vpcd cpu top next-token candidates:")
    for item in vpcd_cpu_next_token_summary["top_tokens"]:
        print(item)
    print("vpcd cloud top next-token candidates:")
    for item in vpcd_next_token_summary["top_tokens"]:
        print(item)
    vpcd_output_comparison
elif ENABLE_VPCD:
    print('Skipping VPCD output inspection because ENABLE_DEBUG_OUTPUT_INSPECTION is False.')
else:
    print('Skipping VPCD cell 30 because ENABLE_VPCD is False.')


### VPCD Hybrid E2E Run

Run this section only after the compiled target has already been resolved.
This is the first point where the notebook executes the real Phase 3 hybrid pipeline:

1. tokenizer encode on the host CPU
2. compiled model-step inference on the cloud NPU target
3. host-side decode loop until EOS or max length
4. write `hybrid-run-<RUN_LABEL>.json` under `build/aihub/records/vpcd_hybrid_option1/`


In [ ]:
if ENABLE_VPCD:
    vpcd_hybrid_report = run_vpcd_hybrid_evaluation(
        runtime_config=RUNTIME_CONFIG,
        run_label=RUN_LABEL,
        explicit_target_model_id=VPCD_TARGET_MODEL_ID,
        max_samples=VPCD_HYBRID_MAX_SAMPLES,
    )
    vpcd_hybrid_record_path = vpcd_hybrid_report["record_path"]

    print("vpcd hybrid target model id:", vpcd_hybrid_report["target_reference"].target_model_id)
    print("vpcd hybrid summary:", vpcd_hybrid_report["summary"])
    print("vpcd hybrid record:", vpcd_hybrid_record_path)
else:
    print('Skipping VPCD cell 32 because ENABLE_VPCD is False.')


### VPCD Final Compare Against Gold Samples

This is the final correctness gate for VPCD in this notebook.
Only this section decides whether the evaluated samples match `golden_samples.jsonl` end to end.


In [ ]:
if ENABLE_VPCD:
    vpcd_hybrid_results = vpcd_hybrid_report["results"]
    vpcd_hybrid_mismatches = [row for row in vpcd_hybrid_results if not row["matches_expected"]]

    print("vpcd final punctuation compare:")
    for row in vpcd_hybrid_results:
        print(
            {
                "sample_index": row["sample_index"],
                "raw_text": row["raw_text"],
                "text": row["text"],
                "expected_text": row["expected_text"],
                "matches_expected": row["matches_expected"],
                "decode_steps": row["decode_steps"],
                "generated_ids": row["generated_ids"],
                "golden_input_ids": row["golden_input_ids"],
                "cloud_inference_seconds": row["cloud_inference_seconds"],
                "decode_seconds": row["decode_seconds"],
            }
        )

    if vpcd_hybrid_mismatches:
        print("vpcd mismatches:")
        for row in vpcd_hybrid_mismatches:
            print(
                {
                    "sample_index": row["sample_index"],
                    "raw_text": row["raw_text"],
                    "text": row["text"],
                    "expected_text": row["expected_text"],
                    "generated_ids": row["generated_ids"],
                }
            )
    else:
        print("vpcd all evaluated samples matched golden outputs.")
else:
    print('Skipping VPCD cell 34 because ENABLE_VPCD is False.')


## After The Notebook Runs

This notebook leaves behind the minimum evidence trail for Phase 2 and Phase 3 reruns.
Use one stable `RUN_LABEL` per compiled artifact set when you want later runs to reuse compile records.


In [ ]:
print("runtime record root:", RUNTIME_CONFIG.record_root)
if ENABLE_ZIPFORMER:
    print("zipformer prepared record:", globals().get("zipformer_prepared_record_path"))
    print("zipformer compile record:", RUNTIME_CONFIG.pilot_record_dir("zipformer_encoder_option1") / f"compile-run-{RUN_LABEL}.json")
    print("zipformer live record:", RUNTIME_CONFIG.pilot_record_dir("zipformer_encoder_option1") / f"live-run-{RUN_LABEL}.json")
    print("zipformer hybrid record:", globals().get("zipformer_hybrid_record_path"))
if ENABLE_VPCD:
    print("vpcd prepared record:", globals().get("vpcd_prepared_record_path"))
    print("vpcd compile record:", RUNTIME_CONFIG.pilot_record_dir("vpcd_option1") / f"compile-run-{RUN_LABEL}.json")
    print("vpcd live record:", RUNTIME_CONFIG.pilot_record_dir("vpcd_option1") / f"live-run-{RUN_LABEL}.json")
    print("vpcd hybrid record:", globals().get("vpcd_hybrid_record_path"))
